In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import wandb
from wandb.integration.keras import WandbCallback, WandbEvalCallback, WandbMetricsLogger



I0000 00:00:1775507906.823190  105672 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# 01 Task

# With tensorflow

In [2]:
# load data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Train:", x_train.shape)
print("Test:", x_test.shape)

/usr/local/lib/python3.11/dist-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train: (50000, 32, 32, 3)
Test: (10000, 32, 32, 3)


In [3]:
x_train = x_train / 255.0
x_test = x_test / 255.0 #To make the pixels range between 0 and 1  instead of 0 and 255

### first version

In [5]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
wandb.init(
    project="cnn-project-lab-zero",
    name="SGD",
    config={
        "epochs": 1000,
        "batch_size": 32
    }
)

config = wandb.config

In [7]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), input_shape=(32,32,3)),
    layers.LeakyReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3)),
    layers.LeakyReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(64),
    layers.LeakyReLU(),

    layers.Dense(10)  # 10 classes
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1775508044.319382  105672 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7798 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:21:00.0, compute capability: 7.5


In [8]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


In [ ]:
model.fit(
    x_train, y_train,
    epochs=config.epochs,
    batch_size=config.batch_size,
    validation_split=0.15,
    callbacks=[WandbMetricsLogger()]
)
model.save("SGD.keras")

Epoch 1/1000


I0000 00:00:1775508194.765708  106039 service.cc:153] XLA service 0x7fad4002dd60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775508194.765749  106039 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.3.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1775508194.901146  106039 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775508194.952277  106039 cuda_dnn.cc:461] Loaded cuDNN version 91900
I0000 00:00:1775508195.134057  106039 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_690__.28


   3/1329 ━━━━━━━━━━━━━━━━━━━━ 47s 36ms/step - accuracy: 0.1250 - loss: 2.3137  

I0000 00:00:1775508201.227613  106039 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1327/1329 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1064 - loss: 2.2988

I0000 00:00:1775508215.127079  106040 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_690__.28


1329/1329 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - accuracy: 0.1127 - loss: 2.2908 - val_accuracy: 0.1343 - val_loss: 2.2768
Epoch 2/1000
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 32s 17ms/step - accuracy: 0.1657 - loss: 2.2625 - val_accuracy: 0.1933 - val_loss: 2.2449
Epoch 3/1000
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.1972 - loss: 2.2272 - val_accuracy: 0.2165 - val_loss: 2.2057
Epoch 4/1000
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - accuracy: 0.2173 - loss: 2.1821 - val_accuracy: 0.2252 - val_loss: 2.1555
Epoch 5/1000
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 22s 16ms/step - accuracy: 0.2422 - loss: 2.1293 - val_accuracy: 0.2524 - val_loss: 2.1013
Epoch 6/1000
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 38s 14ms/step - accuracy: 0.2696 - loss: 2.0720 - val_accuracy: 0.2719 - val_loss: 2.0453
Epoch 7/1000
1323/1329 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2846 - loss: 2.0307

In [ ]:
score = model.evaluate(x_test, y_test)
print(score)

In [33]:
#new = tf.keras.models.load_model("test.keras")


313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.1730 - loss: 2.2701
[2.2700717449188232, 0.17299999296665192]
{'name': 'SGD', 'learning_rate': 9.999999747378752e-05, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'momentum': 0.0, 'nesterov': False}


AttributeError: 'SGD' object has no attribute 'lr'

In [ ]:
wandb.finish()

### second version - with adam

In [ ]:
wandb.init(
    project="cnn-project-lab-zero",
    name="adam",
    config={
        "epochs": 1000,
        "batch_size": 32
    }
)

config = wandb.config

In [ ]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), input_shape=(32,32,3)),
    layers.LeakyReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3)),
    layers.LeakyReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(64),
    layers.LeakyReLU(),

    layers.Dense(10)  # 10 classes
])

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
model.fit(
    x_train, y_train,
    epochs=config.epochs,
    batch_size=config.batch_size,
    validation_split=0.15,
    callbacks=[WandbMetricsLogger()]
)
model.save("adam.keras")

In [ ]:
score = model.evaluate(x_test, y_test)
print(score)
wandb.finish()

### third version - with tanh

In [16]:
wandb.init(
    project="cnn-project-lab-zero",
    name="tanh",
    config={
        "epochs": 1000,
        "batch_size": 32
    }
)

config = wandb.config

epoch/accuracy,▁▅▇▇████████████████████████████████████
epoch/epoch,▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,██▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
epoch/val_loss,▁▁▁▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
epoch/accuracy,0.99715
epoch/epoch,824
epoch/learning_rate,0.0005
epoch/loss,0.01565
epoch/val_accuracy,0.66813


In [ ]:
# Model
model = models.Sequential([

    layers.Conv2D(32, (3,3), input_shape=(32,32,3)),
    layers.Activation('tanh'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3)),
    layers.Activation('tanh'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(64),
    layers.Activation('tanh'),

    layers.Dense(10)
])

# Optimizer Adam
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(
    x_train, y_train,
    epochs=config.epochs,
    batch_size=config.batch_size,
    validation_split=0.15,
    callbacks=[WandbMetricsLogger()]
)
model.save("tanh.keras")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/1000


I0000 00:00:1775549877.577182  106038 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_11476653__.28


1328/1329 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3889 - loss: 1.7154

I0000 00:00:1775549916.355473  106056 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_11476653__.28


1329/1329 ━━━━━━━━━━━━━━━━━━━━ 65s 44ms/step - accuracy: 0.4634 - loss: 1.5192 - val_accuracy: 0.5369 - val_loss: 1.3212
Epoch 2/1000
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 50s 37ms/step - accuracy: 0.5868 - loss: 1.1830 - val_accuracy: 0.6004 - val_loss: 1.1465
Epoch 3/1000
 410/1329 ━━━━━━━━━━━━━━━━━━━━ 33s 37ms/step - accuracy: 0.6297 - loss: 1.0791

In [ ]:
score = model.evaluate(x_test, y_test)
print(score)
wandb.finish()

## Results

In [ ]:
#test_loss, test_acc = model.evaluate(x_test, y_test)
#print("Accuracy sur test set :", test_acc)